Import Libraries

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

Load & Preprocess MNIST

In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize (0–255 → 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0

# Flatten (28x28 → 784)
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print("Shape:", x_train.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Shape: (60000, 784)


Build Model

In [4]:
model = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(784,)),
    layers.Dense(10)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compile & Train

In [5]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8793 - loss: 0.4352
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9327 - loss: 0.2365
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9410 - loss: 0.2069
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9462 - loss: 0.1895
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9502 - loss: 0.1765


Evaluate

In [6]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9500 - loss: 0.1801
Test accuracy: 0.949999988079071


Extract Weights

In [7]:
W1, b1 = model.layers[0].get_weights()
W2, b2 = model.layers[1].get_weights()

print("W1:", W1.shape)
print("W2:", W2.shape)

W1: (784, 16)
W2: (16, 10)


Export to C Header

In [8]:
def to_c_array(name, arr):
    flat = arr.flatten()
    c_str = f"float {name}[{len(flat)}] = {{"
    c_str += ",".join(map(str, flat))
    c_str += "};\n"
    return c_str

with open("weights.h", "w") as f:
    f.write("#ifndef WEIGHTS_H\n#define WEIGHTS_H\n\n")

    f.write(to_c_array("W1", W1))
    f.write(to_c_array("b1", b1))
    f.write(to_c_array("W2", W2))
    f.write(to_c_array("b2", b2))

    f.write("\n#endif")

Download Weights

In [9]:
from google.colab import files
files.download("weights.h")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>